In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.ticker

d = np.load("../data/processed/orientation_decoder_ready_top1000.npz")
X, y_cos2, y_sin2 = d["X"], d["y_cos2"], d["y_sin2"]
theta_deg, theta_rad = d["theta_deg"], d["theta_rad"]

rng = np.random.default_rng(42)
idx = rng.permutation(X.shape[0])
s = int(0.8 * len(idx))
tr, te = idx[:s], idx[s:]

W, _, _, _ = np.linalg.lstsq(X[tr], np.column_stack([y_cos2[tr], y_sin2[tr]]), rcond=None)
Yp = X[te] @ W
pred_rad = np.arctan2(Yp[:, 1], Yp[:, 0]) / 2 % np.pi
err = (pred_rad - theta_rad[te] + np.pi / 2) % np.pi - np.pi / 2
mae = float(np.degrees(np.abs(err).mean()))
print(f"MAE = {mae:.1f}\u00b0")

In [ ]:
cv = pd.read_csv("../reports/tables/issue3_decoder_comparison.csv")
top = cv[cv["source"] == "top_1000_pool"]
rand = cv[cv["source"] == "random_full_population"]

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.errorbar(rand.neuron_count, rand.mae_mean, yerr=rand.mae_std,
            fmt="o-", color="#2171b5", capsize=4, ms=7, lw=2, label="Random neurons")
ax.errorbar(top.neuron_count, top.mae_mean, yerr=top.mae_std.fillna(0),
            fmt="s-", color="#e6550d", capsize=4, ms=7, lw=2, label="Top-1000 reliable")
ax.axhline(45, color="gray", ls=":", label="Chance")
ax.set_xlabel("Neurons"); ax.set_ylabel("MAE (\u00b0)")
ax.set_xscale("log")
ticks = sorted(set(top.neuron_count.tolist() + rand.neuron_count.tolist()))
ax.set_xticks(ticks)
ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
ax.tick_params(axis="x", rotation=45)
ax.legend(); ax.set_ylim(0, 50)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(np.degrees(theta_rad[te]), np.degrees(pred_rad), s=1, alpha=0.3)
ax.plot([0, 180], [0, 180], "r--", lw=1)
ax.set_xlabel("True (\u00b0)"); ax.set_ylabel("Predicted (\u00b0)")
ax.set_title(f"MAE = {mae:.1f}\u00b0")
ax.set_aspect("equal")
plt.tight_layout()